AI ethics practices are often grounded in simple data science processes around inspecting your dataset and your model. In this exercise, you will implement an example set of technical guardrails for AI systems.

There is no right or wrong way of approaching this exercise - the objective is to get you thinking on how you can move from "inspection" of these processes to implementing technical validation checks and guardrails. A key element that will help you with this exercise is identifying the type of information you would need for your validations.

You're welcome to change the way we pass inputs to the test functions to capture the right dependencies, for example whether that's the training data dataframe itself or a summary of the training data instead.

## Load Data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import datasets, ensemble
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
from sklearn import datasets

diabetes = datasets.load_diabetes(as_frame=True)
X, y = diabetes.data, diabetes.target

## Train and test the performance of an ML Model

In this section, you are asked to train a simple Linear Regression model on the training dataset using scikit-learn and make predictions on the test dataset. Use the MSE to measure the performance of the model.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=15
)

In [ ]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train the model
LR_model = LinearRegression()
LR_model.fit(X_train, y_train)
# Compatibility alias for the earlier model name.
model = LR_model

# Evaluate the model on unseen test data
y_pred = model.predict(X_test)
diabetes_y_pred = y_pred
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test Mean Squared Error: {mse:.2f}")
print(f"Test R² Score: {r2:.3f}")

### Implementing Sample Guardrails

Implement the below functions with tests for validating the data input, model performance, model outputs, and the uncertainity of the model. We have proposed a few sample guidelines for you below. Remember: There is no right or wrong answer - within the context of these guidelines, you may choose your implementation approach and scope as you see relevant.

In [ ]:
#DO NOT MODIFY - creating test data
data_point = X_test.iloc[10]
y_data_point = y_test.iloc[10]

incorrect_data_point = data_point.copy()
y_incorrect_data_point = 5000
incorrect_data_point['bp'] = 6.0
incorrect_model_predictions = np.random.default_rng().uniform(30, 200, 89)
incorrect_pred_range = np.random.default_rng().uniform(500, 800, 89)

#### Validating data inputs

If an unexpected data input that isn't part of the training dataset is passed to a model, we can anticipate unexpected performance of the model, which can impact the robustness of the AI systems.

Implement a check to ensure your input variable is within the minimum and maxiumum limits of the training dataset.

**Hint:** Think about the information you would need to validate input data.

**Hint:** You may choose to modify the ways inputs are passed to the test_model_inputs function. For example, you could pass the training dataset directly to the test_model_inputs() function. Alternatively, you may choose to store the min and max values for the training dataset seperately for your function for the validation.

In [ ]:
def test_model_inputs(data_point, df):
    """Return True when a model input is valid for the training data."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return False

    # Convert a single input row to a Series while preserving feature names.
    if isinstance(data_point, pd.DataFrame):
        if len(data_point) != 1:
            return False
        data_point = data_point.iloc[0]
    elif isinstance(data_point, dict):
        data_point = pd.Series(data_point)
    elif not isinstance(data_point, pd.Series):
        return False

    # Inputs must have exactly the same features as the training data.
    if set(data_point.index) != set(df.columns):
        return False
    data_point = data_point.reindex(df.columns)

    # Reject missing, non-numeric, or out-of-range feature values.
    if data_point.isna().any():
        return False
    try:
        values = pd.to_numeric(data_point)
    except (TypeError, ValueError):
        return False

    lower_bounds = df.min()
    upper_bounds = df.max()
    return bool(((values >= lower_bounds) & (values <= upper_bounds)).all())

Next, test the function on a correct and invalid data point. Your function should succesfully fail the invalid data point and pass the correct data point.

In [ ]:
#DO NOT MODIFY - TRY EXCEPT BLOCKS
try:
    #OPTIONAL: You may choose to modify the inputs to this function
    test_model_inputs(data_point, X_train)
except:
    print("Try again. Your function is failing on the correct data point.")

try:
    #OPTIONAL: You may choose to modify the inputs to this function
    test_model_inputs(incorrect_data_point, X_train)
except ValueError:
    print("Correct! Your function has detected the model input is out of bounds")

#### Test Model Performance

If model performance drops, there can be issues related to the mis-execution of the model and potential bias issues caused by inaccuracies being introduced into the output prediction process.

Create a simple check to ensure the MSE of the model does not bypass a high value you may determine, e.g., 3000.

In [ ]:
def test_model_performance(X, y, model):
    """Return True when the model's MSE is within the allowed limit."""
    mse_threshold = 3000

    if X is None or y is None or model is None:
        return False
    try:
        predictions = model.predict(X)
        mse = mean_squared_error(y, predictions)
    except (AttributeError, TypeError, ValueError):
        return False

    print(f"Model MSE: {mse:.2f}")
    if not np.isfinite(mse) or mse > mse_threshold:
        raise ValueError(
            f"Model performance is outside the allowed MSE limit of {mse_threshold}."
        )
    return True

In [ ]:
#DO NOT MODIFY - TRY EXCEPT BLOCKs
try:
    #OPTIONAL: You may choose to modify the inputs to this function
    test_model_performance(X_test, y_test, LR_model)
except ValueError:
    print("Try again. Your function is failing on the acceptable model predictions.")

try:
    #OPTIONAL: You may choose to modify the inputs to this function
    test_model_performance(X_test, incorrect_model_predictions, LR_model)
except ValueError:
    print("Correct! Your function has detected the model performance is out of bounds")

#### Test Model Outputs

Similar to the input data validation, checking to see that model outputs are in a correct range can help us:

1. Identify incorrect model predictions
2. Identify potentially incorrect data causing the outrageous predictions that weren't caught by our initial checks.

You may choose to test this on the training or test dataset.

In [ ]:
def test_model_outputs(X, LR_model, y_pred):
    """Validate that supplied outputs match finite model predictions."""
    if X is None or LR_model is None or y_pred is None:
        raise ValueError("Inputs and model must be provided.")

    try:
        expected_predictions = np.asarray(LR_model.predict(X), dtype=float)
        supplied_predictions = np.asarray(y_pred, dtype=float)
    except (AttributeError, TypeError, ValueError):
        raise ValueError("Model outputs must be numeric predictions.")

    if (
        expected_predictions.shape != supplied_predictions.shape
        or not np.isfinite(supplied_predictions).all()
        or not np.allclose(supplied_predictions, expected_predictions, rtol=1e-5, atol=1e-8)
    ):
        raise ValueError("Model outputs are invalid or outside the expected range.")

    return True

We are peforming a simple reshape of the data points arrays, hence the "UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names" can be ignored for this exercise.

In [ ]:
#DO NOT MODIFY - TRY EXCEPT BLOCKS
try:
    #OPTIONAL: You may choose to modify the inputs to this function
    test_model_outputs(X_test, LR_model, diabetes_y_pred)
    test_model_outputs(np.array(data_point).reshape(1, -1), LR_model, diabetes_y_pred)
except ValueError:
    print("Try again. Your function is failing on the acceptable model predictions.")

try:
    #OPTIONAL: You may choose to modify the inputs to this function
    test_model_outputs(np.array(data_point).reshape(1, -1), LR_model, incorrect_pred_range)
except ValueError:
    print("Correct! Your function has detected the model performance is out of bounds")

Throughout this exercise, you may have recognized that guardrails are targeting the data and models, and different types of data are required for the validation checks (training data, test data, sample data points, model predictions, model). Keep these considerations in mind as you implement technical guardrails accompanying organizational guardrails for AI Ethics.